[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/supervised/linear_regression_five_ways.ipynb)

# Linear Regression: Five Ways to Solve the Same Problem

**Companion blog post:** [Linear Regression: Five Ways to Solve the Same Problem](https://sesen.ai/blog/linear-regression-five-ways)

---

Linear regression is the "Hello World" of machine learning. Behind this simple model lie five fundamentally different algorithms — each from a different branch of mathematics:

1. **Normal Equation** — Linear algebra (closed-form solution)
2. **Gradient Descent** — Calculus (iterative optimisation)
3. **SVD** — Numerical analysis (pseudo-inverse)
4. **scipy.optimize** — Generic optimisation (black-box solver)
5. **sklearn** — Production one-liner (SVD internally)

They all produce the *exact same answer*. That's the punchline: five roads, one destination.

By the end of this notebook, you'll implement all five from scratch and understand *why* they give the same result.

## Setup

We create a synthetic dataset with a clear linear trend plus Gaussian noise. The true relationship is $y = 2.5x + 7 + \epsilon$, where $\epsilon \sim \mathcal{N}(0, 2.5^2)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

np.random.seed(42)
n = 50
X_raw = np.random.uniform(0, 10, n)
y = 2.5 * X_raw + 7 + np.random.normal(0, 2.5, n)

# Design matrix: add a column of ones for the intercept
X = np.column_stack([np.ones(n), X_raw])  # shape (50, 2)

# Quick look at the data
plt.figure(figsize=(8, 5))
plt.scatter(X_raw, y, alpha=0.7, edgecolors='k', linewidth=0.5, label='Data')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Synthetic Linear Regression Dataset')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f'Dataset: {n} samples')
print(f'Design matrix X shape: {X.shape}')
print(f'True parameters: intercept = 7, slope = 2.5')

Our goal: find $\boldsymbol{\beta} = [\beta_0, \beta_1]$ that minimises the sum of squared residuals:

$$\min_{\boldsymbol{\beta}} \|\mathbf{y} - X\boldsymbol{\beta}\|^2$$

The true values are $\beta_0 = 7$ (intercept) and $\beta_1 = 2.5$ (slope). Let's recover them five ways.

## Method 1: Normal Equation

The closed-form solution from linear algebra. Setting the gradient of the loss to zero gives:

$$\boldsymbol{\beta} = (X^\top X)^{-1} X^\top \mathbf{y}$$

This is **exact** — no iteration, no approximation. But it requires $X^\top X$ to be invertible.

In [ ]:
beta_normal = np.linalg.inv(X.T @ X) @ X.T @ y
print(f'Normal equation: y = {beta_normal[0]:.2f} + {beta_normal[1]:.2f}x')

## Method 2: Gradient Descent

Instead of solving analytically, gradient descent takes small steps in the direction of steepest descent:

$$\boldsymbol{\beta}^{(t+1)} = \boldsymbol{\beta}^{(t)} - \eta \nabla_{\boldsymbol{\beta}} L$$

where $\eta$ is the learning rate and the gradient is $\nabla L = \frac{2}{n}X^\top(X\boldsymbol{\beta} - \mathbf{y})$.

In [ ]:
def gradient_descent(X, y, lr=0.001, n_iter=500):
    beta = np.zeros(X.shape[1])
    history = [beta.copy()]
    loss_history = []

    for _ in range(n_iter):
        residuals = X @ beta - y
        loss = np.mean(residuals ** 2)
        loss_history.append(loss)
        gradient = (2 / len(y)) * X.T @ residuals
        beta -= lr * gradient
        history.append(beta.copy())

    return beta, history, loss_history

beta_gd, gd_history, gd_loss = gradient_descent(X, y, lr=0.02, n_iter=1000)
print(f'Gradient descent: y = {beta_gd[0]:.2f} + {beta_gd[1]:.2f}x')

### Gradient Descent Convergence

Let's visualise how the loss decreases over iterations and watch the regression line converge.

In [ ]:
# Loss curve
plt.figure(figsize=(8, 4))
plt.plot(gd_loss, linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Mean Squared Error')
plt.title('Gradient Descent Convergence (lr=0.02)')
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

In [ ]:
# Animation of gradient descent convergence
x_line = np.linspace(0, 10, 100)

# Select frames: more at the start (where the action is), fewer at the end
frames = list(range(0, 20, 1)) + list(range(20, 100, 5)) + list(range(100, min(len(gd_history), 501), 50))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_raw, y, alpha=0.7, edgecolors='k', linewidth=0.5, zorder=5)
line, = ax.plot([], [], 'r-', linewidth=2, label='GD estimate')
true_line, = ax.plot(x_line, 7 + 2.5 * x_line, 'g--', alpha=0.5, linewidth=1, label='True line')
title = ax.set_title('')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_xlim(-0.5, 10.5)
ax.set_ylim(y.min() - 3, y.max() + 3)
ax.legend()
ax.grid(True, alpha=0.3)

def init():
    line.set_data([], [])
    return line,

def animate(frame_idx):
    i = frames[frame_idx]
    b = gd_history[i]
    y_line = b[0] + b[1] * x_line
    line.set_data(x_line, y_line)
    title.set_text(f'Gradient Descent — Iteration {i}')
    return line, title

anim = FuncAnimation(fig, animate, init_func=init, frames=len(frames), interval=100, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

## Method 3: SVD (Pseudo-Inverse)

The numerically stable route via singular value decomposition. The SVD decomposes $X = U \Sigma V^\top$, and the solution becomes:

$$\boldsymbol{\beta} = V \Sigma^{-1} U^\top \mathbf{y}$$

This avoids computing $X^\top X$ entirely, which matters when $X^\top X$ is ill-conditioned.

In [ ]:
U, s, Vt = np.linalg.svd(X, full_matrices=False)
beta_svd = Vt.T @ np.diag(1 / s) @ U.T @ y
print(f'SVD:             y = {beta_svd[0]:.2f} + {beta_svd[1]:.2f}x')

## Method 4: scipy.optimize.minimize

Treat regression as a generic optimisation problem. We define the MSE loss and let Nelder-Mead (a derivative-free simplex method) find the minimum.

In [ ]:
from scipy.optimize import minimize

def mse_loss(beta, X, y):
    return np.mean((y - X @ beta) ** 2)

result = minimize(mse_loss, x0=[0, 0], args=(X, y), method='Nelder-Mead')
beta_scipy = result.x
print(f'scipy.optimize:  y = {beta_scipy[0]:.2f} + {beta_scipy[1]:.2f}x')

## Method 5: sklearn

The one-liner you'll use in practice. Internally, sklearn uses SVD via `numpy.linalg.lstsq`.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_raw.reshape(-1, 1), y)
beta_sklearn = np.array([model.intercept_, model.coef_[0]])
print(f'sklearn:         y = {beta_sklearn[0]:.2f} + {beta_sklearn[1]:.2f}x')

## Comparison: All Five Methods

Let's overlay all five solutions on the scatter plot. They should produce identical regression lines.

In [ ]:
methods = {
    'Normal Equation': beta_normal,
    'Gradient Descent': beta_gd,
    'SVD': beta_svd,
    'scipy.optimize': beta_scipy,
    'sklearn': beta_sklearn,
}

x_plot = np.linspace(0, 10, 100)
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']
linestyles = ['-', '--', '-.', ':', (0, (3, 1, 1, 1))]

plt.figure(figsize=(10, 6))
plt.scatter(X_raw, y, alpha=0.6, edgecolors='k', linewidth=0.5, zorder=5, label='Data')

for (name, beta), color, ls in zip(methods.items(), colors, linestyles):
    y_plot = beta[0] + beta[1] * x_plot
    plt.plot(x_plot, y_plot, color=color, linestyle=ls, linewidth=2.5,
             label=f'{name}: y = {beta[0]:.2f} + {beta[1]:.2f}x')

plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Five Methods, One Answer', fontsize=14)
plt.legend(fontsize=9, loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()

# Print comparison table
print(f'{"Method":<20} {"Intercept":>10} {"Slope":>10}')
print('=' * 42)
for name, beta in methods.items():
    print(f'{name:<20} {beta[0]:>10.4f} {beta[1]:>10.4f}')
print(f'{"True values":<20} {7.0:>10.4f} {2.5:>10.4f}')

## The Mathematics

### Normal Equation Derivation

The mean squared error is:

$$L(\boldsymbol{\beta}) = \frac{1}{n}\|\mathbf{y} - X\boldsymbol{\beta}\|^2$$

Setting the gradient to zero:

$$\nabla_{\boldsymbol{\beta}} L = -\frac{2}{n} X^\top(\mathbf{y} - X\boldsymbol{\beta}) = \mathbf{0}$$

Solving for $\boldsymbol{\beta}$:

$$X^\top X \boldsymbol{\beta} = X^\top \mathbf{y} \quad \Longrightarrow \quad \boldsymbol{\beta} = (X^\top X)^{-1} X^\top \mathbf{y}$$

The name "normal" comes from geometry: the residual vector $\mathbf{y} - X\boldsymbol{\beta}$ is *normal* (perpendicular) to the column space of $X$.

### Gradient Descent Update Rule

The gradient of the MSE loss is:

$$\nabla_{\boldsymbol{\beta}} L = \frac{2}{n} X^\top(X\boldsymbol{\beta} - \mathbf{y})$$

The update rule subtracts a step in the gradient direction:

$$\boldsymbol{\beta}^{(t+1)} = \boldsymbol{\beta}^{(t)} - \eta \cdot \frac{2}{n} X^\top(X\boldsymbol{\beta}^{(t)} - \mathbf{y})$$

The learning rate $\eta$ controls step size. Too large and we overshoot; too small and convergence is slow.

### SVD Decomposition

The SVD decomposes $X = U \Sigma V^\top$ where:
- $U$ is an $n \times p$ orthogonal matrix (left singular vectors)
- $\Sigma$ is a $p \times p$ diagonal matrix (singular values)
- $V^\top$ is a $p \times p$ orthogonal matrix (right singular vectors)

Substituting into the normal equation:

$$(U\Sigma V^\top)^\top(U\Sigma V^\top)\boldsymbol{\beta} = (U\Sigma V^\top)^\top \mathbf{y}$$

$$V\Sigma^2 V^\top \boldsymbol{\beta} = V\Sigma U^\top \mathbf{y}$$

$$\boldsymbol{\beta} = V \Sigma^{-1} U^\top \mathbf{y}$$

This avoids forming $X^\top X$ and works even when it's ill-conditioned.

## Bonus: Ridge Regression

When $X^\top X$ is singular or nearly so, the normal equation is unstable. **Ridge regression** adds a penalty $\lambda\|\boldsymbol{\beta}\|^2$ to the objective:

$$\boldsymbol{\beta}_{\text{ridge}} = (X^\top X + \lambda I)^{-1} X^\top \mathbf{y}$$

The $\lambda I$ term makes the matrix invertible regardless of collinearity. Larger $\lambda$ shrinks coefficients toward zero — a bias-variance tradeoff.

**Bayesian interpretation:** Ridge regression is equivalent to placing a Gaussian prior $\boldsymbol{\beta} \sim \mathcal{N}(\mathbf{0}, \sigma^2/\lambda \cdot I)$ on the weights and computing the MAP estimate.

In [ ]:
def ridge_regression(X, y, lam):
    """Ridge regression: closed-form solution."""
    n_features = X.shape[1]
    return np.linalg.inv(X.T @ X + lam * np.eye(n_features)) @ X.T @ y

lambdas = [0, 1, 10, 100]

print(f'{"lambda":<10} {"Intercept":>10} {"Slope":>10}')
print('=' * 32)
for lam in lambdas:
    beta_r = ridge_regression(X, y, lam)
    print(f'{lam:<10d} {beta_r[0]:>10.2f} {beta_r[1]:>10.2f}')

In [ ]:
# Visualise Ridge regression for different lambda values
plt.figure(figsize=(10, 6))
plt.scatter(X_raw, y, alpha=0.6, edgecolors='k', linewidth=0.5, zorder=5, label='Data')

ridge_colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
for lam, color in zip(lambdas, ridge_colors):
    beta_r = ridge_regression(X, y, lam)
    y_ridge = beta_r[0] + beta_r[1] * x_plot
    label = f'$\\lambda={lam}$: y = {beta_r[0]:.2f} + {beta_r[1]:.2f}x'
    plt.plot(x_plot, y_ridge, color=color, linewidth=2, label=label)

plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Ridge Regression: Effect of Regularisation', fontsize=14)
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.show()

## Exercises

Try these to deepen your understanding:

In [ ]:
# Exercise 1: Learning Rate Experiments
# Try learning rates of 0.0001, 0.005, and 0.05.
# Plot the loss curve for each. At what rate does gradient descent diverge?
#
# Hint: use the gradient_descent function defined above.

# Your code here:
# learning_rates = [0.0001, 0.005, 0.05]
# for lr in learning_rates:
#     beta, history, loss = gradient_descent(X, y, lr=lr, n_iter=1000)
#     plt.plot(loss, label=f'lr={lr}')
# plt.legend()
# plt.yscale('log')
# plt.show()

In [ ]:
# Exercise 2: L1 vs L2 Regression with Outliers
# Generate data with 3 outliers. Compare the OLS line (L2) with the
# minimum absolute deviation line (L1, using scipy.optimize).
# Which is more robust to outliers?
#
# Hint: define an L1 loss function:
#   def l1_loss(beta, X, y):
#       return np.mean(np.abs(y - X @ beta))

# Your code here:
# np.random.seed(42)
# X_ex = np.column_stack([np.ones(53), np.random.uniform(0, 10, 53)])
# y_ex = 2.5 * X_ex[:, 1] + 7 + np.random.normal(0, 2.5, 53)
# # Add 3 outliers
# y_ex[:3] = [50, 55, 60]
# ...

In [ ]:
# Exercise 3: Polynomial Features
# Generate data from y = x^2 + noise and fit linear regression
# with features [1, x, x^2]. How does the design matrix change?
#
# Hint:
# x_poly = np.random.uniform(-3, 3, 100)
# y_poly = x_poly**2 + np.random.normal(0, 1, 100)
# X_poly = np.column_stack([np.ones(100), x_poly, x_poly**2])
# beta_poly = np.linalg.inv(X_poly.T @ X_poly) @ X_poly.T @ y_poly

# Your code here:

In [ ]:
# Exercise 4: Condition Number Experiments
# Create two nearly collinear features (x2 = x1 + epsilon).
# Compare the normal equation vs SVD solutions.
# What happens to np.linalg.cond(X.T @ X)?
#
# Hint:
# x1 = np.random.uniform(0, 10, 100)
# x2 = x1 + np.random.normal(0, 0.001, 100)  # Nearly collinear!
# X_coll = np.column_stack([np.ones(100), x1, x2])
# print(f'Condition number: {np.linalg.cond(X_coll.T @ X_coll):.2e}')
# # Compare: normal equation vs np.linalg.lstsq (SVD-based)

# Your code here:

## Summary

### Key Takeaways

| Method | Approach | When to Use |
|---|---|---|
| Normal Equation | Closed-form, $O(p^2 n + p^3)$ | Small number of features (< ~10k) |
| Gradient Descent | Iterative, $O(npT)$ | Large datasets, online learning |
| SVD | Pseudo-inverse, $O(np^2)$ | Ill-conditioned problems |
| scipy.optimize | Black-box solver | Custom loss functions |
| sklearn | SVD internally | Production code |

All five methods minimise the same objective — the sum of squared residuals — so they converge to the same $\boldsymbol{\beta}$.

### What's Next?

- **[Gaussian Process Regression](https://sesen.ai/blog/gaussian-process-regression-from-scratch)** — The normal equation generalises to infinite-dimensional feature spaces
- **[Backpropagation](https://sesen.ai/blog/backpropagation-neural-nets-from-first-principles)** — Gradient descent on a much harder problem
- **[From MLE to Bayesian Inference](https://sesen.ai/blog/from-mle-to-bayesian-inference)** — Ridge regression is OLS with a prior

---

**Author:** Dr. Berkan Sesen | [sesen.ai](https://sesen.ai)

**Companion blog post:** [Linear Regression: Five Ways to Solve the Same Problem](https://sesen.ai/blog/linear-regression-five-ways)